# Fixed vs NACK-Adaptive MCS — Comparison

Self-contained notebook reading the five tidy CSVs from each campaign:

- Fixed:    `../traffic_class_eval/results_traffic_class_eval/tidy/`
- Adaptive: `./results_traffic_class_adaptive_eval/tidy/`

Both campaigns use the identical 5-class × 20-seed sweep on the
`intersection_nlosv` trace; the *only* delta is the MCS controller.


## §1. Setup — load both campaigns into a single long-form DataFrame


In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

FIXED_DIR    = Path("../traffic_class_eval/results_traffic_class_eval/tidy")
ADAPTIVE_DIR = Path("./results_traffic_class_adaptive_eval/tidy")

CLASS_ORDER = ["periodic1", "periodic2", "periodic3", "aperiodic1", "aperiodic2"]
SCHEME_COLORS = {"fixed": "#1f77b4", "adaptive": "#d62728"}
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 110

def load_scheme(dir_, scheme):
    out = {}
    for name in ["per_rx_kpis", "time_series", "resource_usage", "harq_stats"]:
        p = dir_ / f"{name}.csv"
        df = pd.read_csv(p) if p.is_file() else pd.DataFrame()
        df["scheme"] = scheme
        out[name] = df
    # adapt_log only exists on the adaptive side
    p = dir_ / "adapt_log.csv"
    out["adapt_log"] = pd.read_csv(p) if p.is_file() else pd.DataFrame()
    return out

F = load_scheme(FIXED_DIR,    "fixed")
A = load_scheme(ADAPTIVE_DIR, "adaptive")

rx = pd.concat([F["per_rx_kpis"],    A["per_rx_kpis"]], ignore_index=True)
ts = pd.concat([F["time_series"],    A["time_series"]], ignore_index=True)
ru = pd.concat([F["resource_usage"], A["resource_usage"]], ignore_index=True)
hq = pd.concat([F["harq_stats"],     A["harq_stats"]], ignore_index=True)
ad = A["adapt_log"]

for df in (rx, ts, ru, hq):
    df["trafficClass"] = pd.Categorical(df["trafficClass"], CLASS_ORDER, ordered=True)
    df["scheme"]       = pd.Categorical(df["scheme"], ["fixed", "adaptive"], ordered=True)

classes = [c for c in CLASS_ORDER if c in rx["trafficClass"].unique()]
print(f"fixed:    rx={len(F['per_rx_kpis'])}  hq={len(F['harq_stats'])}")
print(f"adaptive: rx={len(A['per_rx_kpis'])}  hq={len(A['harq_stats'])}  ad={len(ad)}")


## §2. Per-flow comparison — PRR / PIR / throughput

Violin + strip per `(trafficClass, scheme)`.  Δ-mean annotation shows
`adaptive − fixed` so the headline benefit is read off directly.


In [ ]:
def delta_means(df, col):
    out = []
    for c in classes:
        m_fix = df[(df["trafficClass"]==c)&(df["scheme"]=="fixed")][col].mean()
        m_ada = df[(df["trafficClass"]==c)&(df["scheme"]=="adaptive")][col].mean()
        out.append((c, m_fix, m_ada, m_ada - m_fix))
    return out

metrics = [("prr","Group PRR", (0,1.02)),
           ("pir_ms","PIR [ms]", None),
           ("tput_kbps","Throughput [kbps]", None)]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (col,label,ylim) in zip(axes, metrics):
    sns.violinplot(data=rx, x="trafficClass", y=col, hue="scheme",
                   palette=SCHEME_COLORS, split=True, inner="quartile",
                   cut=0, ax=ax, order=classes)
    sns.stripplot(data=rx, x="trafficClass", y=col, hue="scheme",
                  dodge=True, color="0.15", size=2, alpha=0.4,
                  ax=ax, order=classes, legend=False)
    ax.set_title(label); ax.set_xlabel("")
    if ylim: ax.set_ylim(ylim)
    # Δ-mean annotation
    deltas = delta_means(rx, col)
    for i, (c, mf, ma, d) in enumerate(deltas):
        if np.isnan(d): continue
        txt = f"Δ={d:+.2f}" if abs(d) < 100 else f"Δ={d:+.0f}"
        color = "g" if (d > 0 and col!="pir_ms") or (d < 0 and col=="pir_ms") else "r"
        ax.annotate(txt, xy=(i, ax.get_ylim()[1]*0.97),
                    xycoords="data", ha="center", va="top",
                    color=color, fontsize=9, fontweight="bold")
    for t in ax.get_xticklabels(): t.set_rotation(20)
    if ax is not axes[0]:
        ax.get_legend().remove()
fig.suptitle("Fixed vs Adaptive MCS — per-class KPI distributions (Δ = adaptive − fixed)", y=1.02)
fig.tight_layout(); plt.show()


## §3. Per-RX comparison

Heatmap of mean PRR (rows = RX node 1..8, columns = scheme) per class, plus a
delta-PRR bar showing where adaptation helps most.


In [ ]:
fig, axes = plt.subplots(2, len(classes), figsize=(3.2 * len(classes), 7),
                          gridspec_kw={"height_ratios": [2.2, 1]})
if len(classes) == 1: axes = axes.reshape(2, 1)

for j, c in enumerate(classes):
    # heatmap (rx × scheme)
    sub = rx[rx["trafficClass"] == c]
    piv = sub.groupby(["rx_node_id", "scheme"])["prr"].mean().unstack()
    piv = piv.reindex(columns=["fixed", "adaptive"])
    ax = axes[0, j]
    im = ax.imshow(piv.values, vmin=0, vmax=1, cmap="RdYlGn", aspect="auto")
    ax.set_title(c, fontsize=10)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["fixed", "adaptive"], fontsize=8)
    ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index, fontsize=8)
    if j == 0: ax.set_ylabel("RX node id")
    # annotate cells
    for ri, rid in enumerate(piv.index):
        for ci, sch in enumerate(["fixed", "adaptive"]):
            v = piv.loc[rid, sch] if sch in piv.columns else np.nan
            if pd.notna(v):
                ax.text(ci, ri, f"{v:.2f}", ha="center", va="center",
                        color="white" if v < 0.5 else "black", fontsize=7)

    # Δ-PRR bar (adaptive − fixed) per RX
    ax = axes[1, j]
    if "fixed" in piv.columns and "adaptive" in piv.columns:
        delta = piv["adaptive"] - piv["fixed"]
        colors = ["g" if d > 0 else "r" for d in delta.values]
        ax.barh(range(len(delta)), delta.values, color=colors, alpha=0.75)
        ax.axvline(0, color="k", lw=0.6)
        ax.set_yticks(range(len(delta))); ax.set_yticklabels(delta.index, fontsize=8)
        ax.set_xlim(-1, 1)
        if j == 0: ax.set_ylabel("RX node id")
        ax.set_xlabel("Δ PRR\n(adaptive − fixed)", fontsize=8)
    else:
        ax.set_visible(False)
fig.suptitle("Per-RX PRR — Fixed (left) vs Adaptive (right), Δ below", y=1.01)
fig.tight_layout(); plt.show()


## §4. HARQ comparison


In [ ]:
metrics = [("nack_rate", "PSFCH NACK rate"),
           ("retx_ratio", "MAC retx ratio"),
           ("phy_err_rate", "PHY decode-error rate"),
           ("timeout_count", "HARQ timeouts")]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, (col, label) in zip(axes, metrics):
    sns.barplot(data=hq, x="trafficClass", y=col, hue="scheme",
                palette=SCHEME_COLORS, ax=ax, order=classes,
                errorbar=("ci", 95))
    ax.set_title(label); ax.set_xlabel("")
    for t in ax.get_xticklabels(): t.set_rotation(20)
    if ax is not axes[0] and ax.get_legend() is not None:
        ax.get_legend().remove()
fig.suptitle("HARQ stats: Fixed vs Adaptive (95% CI)", y=1.02)
fig.tight_layout(); plt.show()


## §5. Resource-usage comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# rbLen mean per (class, scheme)
ax = axes[0]
sns.barplot(data=ru, x="trafficClass", y="rbLen", hue="scheme",
            palette=SCHEME_COLORS, ax=ax, order=classes, errorbar=("ci",95))
ax.set_title("Mean PRBs per grant"); ax.set_xlabel("")
for t in ax.get_xticklabels(): t.set_rotation(20)

# Grants per seed
ax = axes[1]
grants = (ru.groupby(["trafficClass", "scheme", "seed"]).size()
            .reset_index(name="n_grants"))
sns.barplot(data=grants, x="trafficClass", y="n_grants", hue="scheme",
            palette=SCHEME_COLORS, ax=ax, order=classes, errorbar=("ci",95))
ax.set_title("Grants per run"); ax.set_xlabel("")
for t in ax.get_xticklabels(): t.set_rotation(20)
if ax.get_legend() is not None: ax.get_legend().remove()
fig.suptitle("Resource usage: Fixed vs Adaptive", y=1.02)
fig.tight_layout(); plt.show()


## §6. Adaptive-only diagnostics (MCS trajectory + decision mix)


In [ ]:
if ad.empty:
    print("adapt_log.csv missing or empty — skipping diagnostics.")
else:
    ad["trafficClass"] = pd.Categorical(ad["trafficClass"], CLASS_ORDER, ordered=True)
    fig, axes = plt.subplots(len(classes), 1, figsize=(11, 2.0*len(classes)),
                              sharex=True, sharey=True)
    if len(classes) == 1: axes = [axes]
    for ax, c in zip(axes, classes):
        sub = ad[ad["trafficClass"] == c]
        for seed, ss in sub.groupby("seed"):
            ax.plot(ss["t_s"], ss["new_mcs"], color="0.6", lw=0.5, alpha=0.4)
        mean = sub.groupby(sub["t_s"].round(1))["new_mcs"].mean()
        ax.plot(mean.index, mean.values, color="#d62728", lw=2.2, label="mean")
        ax.axhline(14, color="k", lw=0.8, ls="--", alpha=0.5, label="MCS=14 (fixed)")
        ax.set_ylabel(f"{c}\nMCS"); ax.set_ylim(-0.5, 28.5)
        ax.legend(loc="upper right", fontsize=8)
    axes[-1].set_xlabel("simulation time [s]")
    fig.suptitle("Adaptive-MCS trajectory per class", y=1.01)
    fig.tight_layout(); plt.show()


## §7. Summary table — the deliverable

Rows: `(trafficClass × scheme)`.  Columns: PRR mean & p5, PIR p95, throughput
p5, MAC retx ratio, mean controller MCS (NaN for the fixed scheme).


In [ ]:
def p5(x):  return np.nanpercentile(x, 5)
def p95(x): return np.nanpercentile(x, 95)

base = (rx.groupby(["trafficClass", "scheme"], observed=True)
          .agg(prr_mean      = ("prr","mean"),
               prr_p5        = ("prr", p5),
               pir_p95_ms    = ("pir_ms", p95),
               tput_p5_kbps  = ("tput_kbps", p5)))
hq_agg = (hq.groupby(["trafficClass", "scheme"], observed=True)
            ["retx_ratio"].mean().rename("retx_ratio_mean"))
summary = base.join(hq_agg)

if not ad.empty:
    mcs_mean = (ad.groupby(["trafficClass"], observed=True)["new_mcs"].mean()
                  .rename("mean_mcs"))
    # Attach to adaptive rows only
    summary["mean_mcs"] = np.nan
    for c, v in mcs_mean.items():
        summary.loc[(c, "adaptive"), "mean_mcs"] = v
else:
    summary["mean_mcs"] = np.nan

summary.round(3).reindex(
    [(c, s) for c in classes for s in ["fixed", "adaptive"]])


**Headline reading**

- Look at the `prr_mean` and `prr_p5` deltas (adaptive − fixed) per class:
  - Periodic3 / Aperiodic2: fixed≈0/0.15.  Any positive Δ here means the
    controller successfully recovered some delivery on large-packet flows.
  - Periodic1 / Aperiodic1: fixed≈0.94/0.86.  Δ should be near zero — these
    flows don't need adaptation.
- `retx_ratio_mean`: adaptive should show higher retx on the hard classes
  (proof it stayed in low MCS where re-transmissions actually fit).
- `mean_mcs` (adaptive rows): MCS the controller settled into; deviation
  from 14 reveals what "the right MCS" for each traffic class looks like
  in this NLOSv scenario.
